# RAG on Azure — Day 5: Conversational RAG (Multi-Turn)

**Use case:** an assistant that holds a real conversation. Ask *"What's the PTO policy?"*, then *"What about for new hires?"* and the second question (which makes no sense on its own) still retrieves the right chunks.

- A **chat history** — the assistant remembers previous turns.
- **Query condensation** — before retrieving, an LLM rewrites the follow-up into a **standalone question** using the history. Pronouns and elided references ("it", "for new hires") get resolved.
- A small **ChatSession** class so you can carry on a real conversation.
- A **topic-switch test** — if you change subjects, the rewriter should *drop* the prior context rather than carry it forward.

## 0. Install dependencies

In [ ]:
%pip install -q "openai>=1.55.3" "azure-search-documents>=11.5.1" python-dotenv

## 1. Load configuration

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

explicit = Path(r"C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\.env")
env_path = explicit if explicit.exists() else (Path(find_dotenv()) if find_dotenv() else None)

if env_path and Path(env_path).exists():
    for line in Path(env_path).read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")
    load_dotenv(env_path, override=False)
else:
    raise FileNotFoundError("No .env file found.")

AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
CHAT_DEPLOYMENT          = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o")
EMBEDDING_DEPLOYMENT     = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_SEARCH_ENDPOINT    = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY     = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME               = os.getenv("AZURE_SEARCH_INDEX", "rag-handbook-day2")

for _n, _v in [("AZURE_OPENAI_ENDPOINT", AZURE_OPENAI_ENDPOINT),
               ("AZURE_OPENAI_API_KEY",  AZURE_OPENAI_API_KEY),
               ("AZURE_SEARCH_ENDPOINT", AZURE_SEARCH_ENDPOINT),
               ("AZURE_SEARCH_API_KEY",  AZURE_SEARCH_API_KEY)]:
    assert _v, f"Set {_n} in your .env"

print("Config OK. Using index:", INDEX_NAME)

Config OK. Using index: ragpipeline-index


## 2. Clients

In [2]:
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

aoai = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)
search_cred = AzureKeyCredential(AZURE_SEARCH_API_KEY)
search_client = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=search_cred)

count = search_client.get_document_count()
print(f"Connected to '{INDEX_NAME}' — {count} documents.")
assert count > 0, "Index is empty — run Day 2 first."

Connected to 'ragpipeline-index' — 10 documents.


## 3. Retrieval

In [4]:
from azure.search.documents.models import VectorizedQuery

def embed(text):
    return aoai.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=[text]).data[0].embedding

def retrieve(query, k=5):
    vq = VectorizedQuery(vector=embed(query), k_nearest_neighbors=k, fields="contentVector")
    results = search_client.search(search_text=None, vector_queries=[vq], select=["content", "source"])
    return [
        {"content": r["content"], "source": r["source"], "score": r["@search.score"]}
        for r in results
    ]

## 4. Query condensation — the new idea

The retriever doesn't know what "new hires" refers to in *"What about for new hires?"* — there's no antecedent in the query itself. So before retrieving, we ask an LLM to **rewrite** the follow-up into a standalone question that combines what the user said with what the conversation has already established.

The prompt below is the heart of conversational RAG. A few rules make it reliable:

1. If the follow-up is already standalone, return it unchanged (no LLM-induced drift).
2. Resolve pronouns and elided references using the history.
3. If the user switches topic, **don't** drag the old topic into the new question.
4. Output only the rewritten question — no preamble.

The one-shot example below teaches all three rules at once.

In [5]:
CONDENSE_SYSTEM = (
    "You are a query rewriter for a question-answering system over company documents. "
    "Given a chat history and a follow-up question, output a STANDALONE question that "
    "can be retrieved against a knowledge base without the chat history.\n\n"
    "Rules:\n"
    "- If the follow-up is already standalone, return it unchanged.\n"
    "- Resolve pronouns ('it', 'they') and elided references ('what about for new hires') using the history.\n"
    "- If the follow-up switches topic, DO NOT inherit context from the prior topic.\n"
    "- Output ONLY the rewritten question. No preamble, no explanation, no quotes."
)

CONDENSE_EXAMPLE = (
    "History:\n"
    "User: What is the PTO policy?\n"
    "Assistant: Full-time employees get 15 days of paid time off per year.\n\n"
    "Follow-up: What about for new hires?\n"
    "Standalone: When do new hires start accruing PTO?"
)

def condense_query(history, question):
    """Rewrite a follow-up question into a standalone one using chat history."""
    if not history:
        return question

    history_text = "\n".join(
        f"User: {turn['user']}\nAssistant: {turn['assistant']}" for turn in history
    )
    user_msg = (
        f"{CONDENSE_EXAMPLE}\n\n---\n\n"
        f"History:\n{history_text}\n\n"
        f"Follow-up: {question}\n"
        f"Standalone:"
    )

    resp = aoai.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": CONDENSE_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

## 5. The ChatSession — putting it all together

Each turn:
1. **Condense** the follow-up into a standalone query (using history).
2. **Retrieve** with that standalone query.
3. **Generate** the answer from retrieved chunks.
4. **Append** to history so the next turn can use it.

We deliberately retrieve with the *condensed* query but show the answer to the user as a response to their *original* question that they shouldn't have to think about the rewriting.

In [6]:
ANSWER_SYSTEM = (
    "You are a helpful assistant answering questions about company policy documents. "
    "Answer ONLY using the numbered context passages. Cite passage numbers like [1], [2]. "
    "If the answer is not in the context, say you don't know based on the documents."
)

class ChatSession:
    def __init__(self, k=5, max_history_turns=5):
        self.history = []
        self.k = k
        self.max_history_turns = max_history_turns   # truncate to avoid prompt bloat

    def ask(self, question, verbose=True):
        # 1) Condense follow-up into a standalone question
        standalone = condense_query(self.history[-self.max_history_turns:], question)

        # 2) Retrieve using the standalone question
        hits = retrieve(standalone, k=self.k)
        context = "\n\n".join(
            f"[{i+1}] (source: {h['source']})\n{h['content']}" for i, h in enumerate(hits)
        )

        # 3) Generate the answer from retrieved chunks
        resp = aoai.chat.completions.create(
            model=CHAT_DEPLOYMENT,
            messages=[
                {"role": "system", "content": ANSWER_SYSTEM},
                {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {standalone}"},
            ],
            temperature=0,
        )
        answer = resp.choices[0].message.content.strip()

        # 4) Append to history
        self.history.append({"user": question, "assistant": answer})

        if verbose:
            print(f"You:        {question}")
            if standalone != question:
                print(f"  (rewritten): {standalone}")
            print(f"Assistant:  {answer}")
            print(f"  sources:  {sorted({h['source'] for h in hits})}")
            print()

        return {"original": question, "standalone": standalone, "answer": answer, "hits": hits}

## 6. Demo — a real multi-turn conversation

Watch the `(rewritten):` lines , that's the condensation step doing its job. The follow-ups *"What about for new hires?"* and *"And can I roll them over?"* are meaningless to a retriever on their own. After condensation they become precise, standalone queries.

In [7]:
chat = ChatSession()

chat.ask("What is the PTO policy?")
chat.ask("What about for new hires?")
chat.ask("And can I roll over unused days?")

You:        What is the PTO policy?
Assistant:  The Paid Time Off (PTO) policy at Acme Corp is as follows:

1. **Annual PTO Allowance**: Full-time employees accrue 15 days of PTO per calendar year, at a rate of 1.25 days per month of completed service. Part-time employees working at least 20 hours per week accrue PTO on a prorated basis based on their scheduled hours. After five years of continuous service, the annual PTO allowance for full-time employees increases to 20 days per year [2].

2. **PTO Accrual for New Hires**: New hires do not accrue PTO during their 90-day probation period. Once the probation period is complete, PTO accrual is applied retroactively to the employee's start date, ensuring no earned time is lost [1], [2].

3. **Carryover**: Employees may carry over a maximum of 5 unused PTO days into the following calendar year. Any balance above 5 days that is unused as of December 31 is forfeited and is not paid out [1].

4. **Requesting PTO**: PTO requests must be submit

{'original': 'And can I roll over unused days?',
 'standalone': 'Can employees roll over unused PTO days into the next calendar year?',
 'answer': 'Yes, employees may carry over a maximum of 5 unused PTO days into the following calendar year. Any balance above 5 days that is unused as of December 31 is forfeited and is not paid out [1].',
 'hits': [{'content': "probation (introductory) period. No\nPTO is accrued during the first 90 days of employment. Once the probation period is complete,\naccrual is applied retroactively to the employee's start date, so no earned time is lost.\n4. Carryover\nEmployees may carry over a maximum of 5 unused PTO days into the following calendar year.\nAny balance above 5 days that is unused as of December 31 is forfeited and is not paid out.\n5. Requesting PTO\nPTO requests should be submitted through the HR portal at least two weeks in advance and\nrequire manager approval. Approval is based on business needs and team coverage.\n6. Sick Leave\nSick leav

## 7. Topic-switch test

A naive history-aware system can over-condense bydragging PTO context into a question about something unrelated. The condense prompt has an explicit rule against this. Below we switch topics mid-conversation and confirm the rewriter doesn't bleed the old topic in.

In [8]:
chat.ask("Switching topics — can I work remotely?")
chat.ask("How many days per week?")
chat.ask("Are there any rules around it?")

You:        Switching topics — can I work remotely?
  (rewritten): Can I work remotely?
Assistant:  I don't know based on the documents. Whether you can work remotely depends on your role's eligibility and your manager's approval, as outlined in the Remote Work Policy [1], [3].
  sources:  ['02_New_Hire_Onboarding_Guide.pdf', '03_Remote_Work_Policy.pdf', '04_Employee_Benefits_Overview.pdf']

You:        How many days per week?
  (rewritten): How many days per week can employees work remotely?
Assistant:  Employees may work remotely up to 3 days per week with manager approval [1].
  sources:  ['01_PTO_and_Leave_Policy.pdf', '02_New_Hire_Onboarding_Guide.pdf', '03_Remote_Work_Policy.pdf']

You:        Are there any rules around it?
  (rewritten): Are there any rules around working remotely?
Assistant:  Yes, there are several rules around working remotely as outlined in the Remote Work Policy:

1. **Eligibility and Approval**: Remote work is permitted for roles that can be performed effec

{'original': 'Are there any rules around it?',
 'standalone': 'Are there any rules around working remotely?',
 'answer': 'Yes, there are several rules around working remotely as outlined in the Remote Work Policy:\n\n1. **Eligibility and Approval**: Remote work is permitted for roles that can be performed effectively off-site, but it requires prior approval from the manager. Certain roles, such as facilities and on-site lab positions, are not eligible for remote work [2].\n\n2. **Days Allowed**: Employees may work remotely up to 3 days per week with manager approval. At least 2 days per week must be worked on-site unless a fully remote arrangement has been formally approved by a department head [2].\n\n3. **Core Hours**: Employees must be reachable and available during core hours of 10:00 am to 3:00 pm local time, regardless of location. They must also attend scheduled meetings via video when working remotely [1], [2].\n\n4. **Equipment and Security**: Acme Corp provides a company lapt

## 8. Inspect the history

The session keeps a log you can inspect, persist, or send back to a UI.

In [9]:
for i, turn in enumerate(chat.history, 1):
    print(f"Turn {i}")
    print(f"  user:      {turn['user']}")
    print(f"  assistant: {turn['assistant'][:100]}{'...' if len(turn['assistant']) > 100 else ''}")
    print()

Turn 1
  user:      What is the PTO policy?
  assistant: The Paid Time Off (PTO) policy at Acme Corp is as follows:

1. **Annual PTO Allowance**: Full-time e...

Turn 2
  user:      What about for new hires?
  assistant: New hires at Acme Corp do not accrue PTO during their first 90 days of employment, which is the prob...

Turn 3
  user:      And can I roll over unused days?
  assistant: Yes, employees may carry over a maximum of 5 unused PTO days into the following calendar year. Any b...

Turn 4
  user:      Switching topics — can I work remotely?
  assistant: I don't know based on the documents. Whether you can work remotely depends on your role's eligibilit...

Turn 5
  user:      How many days per week?
  assistant: Employees may work remotely up to 3 days per week with manager approval [1].

Turn 6
  user:      Are there any rules around it?
  assistant: Yes, there are several rules around working remotely as outlined in the Remote Work Policy:

1. **El...



## Day 5 checklist

1. Follow-ups like *"What about for new hires?"* get rewritten into standalone questions
2. The rewritten query retrieves the right chunks (you can see it in the printed sources)
3. Topic switches don't carry old context into the new question
4. History accumulates across turns and can be inspected

### Learning:
- **Retrieval needs full questions; users speak in fragments.** Condensation bridges that gap.
- **Two LLM calls per turn** (condense + answer) is the standard pattern the extra cost is worth the conversational quality.
- **History grows unbounded.** In production, truncate to the last N turns (we do this via `max_history_turns`) or summarise older history.